# Dashboard Interativo CFD

Este dashboard interativo permite explorar o acoplamento das equações de transporte para $\phi_1$ e $\phi_2$ de forma transiente, observando a animação temporal da reação desde a condição inicial até um tempo final `t_final` arbitrado.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
from ipywidgets import interact_manual, FloatSlider, Dropdown

# Adiciona a pasta src ao path
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from malha import Malha
from solver import TransportSolver

# Carrega a malha (mesma de main.py)
data_path = os.path.join(os.getcwd(), '..', 'data', 'LidDrivenCavityFlow_Re_1000.dat')
grid = Malha(data_path, Nx=40, Ny=40, interp_method='linear')

# Condições de contorno
bc_phi1 = {'west': 0.0, 'east': 0.0, 'south': 1.0, 'north': 0.0}
bc_phi2 = {'west': 0.0, 'east': 0.0, 'south': 0.0, 'north': 1.0}

def simular_e_animar(esquema, k, t_final, dt):
    # Fechar figuras anteriores para não acumular
    plt.close('all')
    
    # Instancia solver com esquema escolhido
    solver = TransportSolver(grid, scheme=esquema, gamma=0.001)
    
    # Resolve transiente para Reagente (phi_2)
    phi2_hist = solver.resolver_transiente(bc_phi2, t_final=t_final, dt=dt, k=k, scalar_type='phi2')
    
    # Resolve transiente para Produto (phi_1), usando o histórico de phi2
    phi1_hist = solver.resolver_transiente(bc_phi1, t_final=t_final, dt=dt, k=k, scalar_type='phi1', phi2_hist=phi2_hist)
    
    # Prepara a figura para a animação
    fig, axs = plt.subplots(1, 2, figsize=(14, 6))
    X, Y = np.meshgrid(grid.x_centros, grid.y_centros, indexing='ij')
    U_centros = 0.5 * (grid.u_faces[:-1, :] + grid.u_faces[1:, :])
    V_centros = 0.5 * (grid.v_faces[:, :-1] + grid.v_faces[:, 1:])
    
    # Determina limites para colormaps
    vmax2 = max(1e-5, max([np.max(f) for f in phi2_hist]))
    vmax1 = max(1e-5, max([np.max(f) for f in phi1_hist]))
    
    def update(frame):
        for ax in axs:
            ax.clear()
            
        # Contorno Phi2
        c1 = axs[0].contourf(X, Y, phi2_hist[frame], levels=25, cmap='plasma', vmin=0, vmax=vmax2)
        axs[0].streamplot(grid.x_centros, grid.y_centros, U_centros.T, V_centros.T, color='white', density=0.8, linewidth=0.5)
        axs[0].set_title(f'Reagente ($\phi_2$) - {esquema} - t={frame*dt:.2f}s')
        axs[0].set_aspect('equal')
        
        # Contorno Phi1
        c2 = axs[1].contourf(X, Y, phi1_hist[frame], levels=25, cmap='plasma', vmin=0, vmax=vmax1)
        axs[1].streamplot(grid.x_centros, grid.y_centros, U_centros.T, V_centros.T, color='white', density=0.8, linewidth=0.5)
        axs[1].set_title(f'Produto ($\phi_1$) - {esquema} - t={frame*dt:.2f}s')
        axs[1].set_aspect('equal')
        
    anim = animation.FuncAnimation(fig, update, frames=len(phi2_hist), interval=100)
    plt.close(fig) # Fecha a figure para não exibir plot estático abaixo da animação
    
    return HTML(anim.to_jshtml())

display(interact_manual(simular_e_animar, 
                esquema=Dropdown(options=['UDS', 'CDS'], value='UDS', description='Esquema:'),
                k=FloatSlider(value=0.00, min=0.00, max=5.0, step=0.01, description='k (Reação):'),
                t_final=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='t_final (s):'),
                dt=FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description='dt (s):')))

interactive(children=(Dropdown(description='Esquema:', options=('UDS', 'CDS'), value='UDS'), FloatSlider(value…

<function __main__.simular_e_animar(esquema, k, t_final, dt)>